# 面试问题：Task Arithmetic、TIES 和 DARE 模型合并分别解决什么问题？

可以直接复述的回答是：第一，把每个微调模型减去共同 base 得到 task delta。第二，Task Arithmetic 直接缩放相加，简单但冲突维度可能放大。第三，TIES 先裁剪小更新，再按坐标选主导符号，只合并同向更新。第四，DARE 随机丢弃 delta 并重标定，用稀疏化缓解冗余。第五，合并前后都要检查参数范数、任务探针和安全回归。第六，模型合并不能替代多任务训练和真实评测。下面用三个客服 Adapter 的可读参数探针演示。

## 真实案例：合并语气、发票抽取与欺诈谨慎三个 Adapter

三个脱敏 Adapter 共享同一个 10 参数教学 base，分别针对友好客服、发票字段抽取和高风险交易谨慎回复。参数名对应可理解能力方向，delta 是人工构造的小向量。真实模型会有数十亿参数，本例只解释坐标冲突与合并门禁，不能推断线上能力。

In [1]:
import numpy as np  # 使用 NumPy 实现 task delta、裁剪和稀疏重标定
parameter_names = ["tone", "brevity", "invoice_id", "amount", "risk", "refusal", "citation", "format", "tool_use", "verbosity"]  # 定义十个可解释参数方向
base = np.zeros(len(parameter_names))  # 使用零向量表示三个 Adapter 的共同基础模型
deltas = {  # 定义三个业务 Adapter 相对 base 的参数变化
    "friendly_support": np.array([0.8, 0.6, 0.0, 0.0, 0.2, -0.2, 0.1, 0.2, 0.0, -0.1]),  # 强化语气和简洁但降低拒绝倾向
    "invoice_extract": np.array([0.1, -0.5, 0.9, 0.8, 0.0, 0.1, 0.4, 0.7, 0.3, -0.2]),  # 强化字段、金额和格式
    "fraud_caution": np.array([0.3, 0.2, -0.2, 0.0, 0.9, 0.8, 0.5, -0.1, 0.4, 0.3]),  # 强化风险、拒绝和引用
}  # 结束三个 task delta
print("Adapter 参数变化：parameter | friendly | invoice | fraud")  # 输入预览展示冲突坐标而非匿名向量
for index, name in enumerate(parameter_names):  # 逐参数输出三个任务更新
    print(f"{name:10} | {deltas['friendly_support'][index]:6.2f} | {deltas['invoice_extract'][index]:6.2f} | {deltas['fraud_caution'][index]:6.2f}")  # 观察 brevity、invoice_id 和 refusal 的差异


Adapter 参数变化：parameter | friendly | invoice | fraud
tone       |   0.80 |   0.10 |   0.30
brevity    |   0.60 |  -0.50 |   0.20
invoice_id |   0.00 |   0.90 |  -0.20
amount     |   0.00 |   0.80 |   0.00
risk       |   0.20 |   0.00 |   0.90
refusal    |  -0.20 |   0.10 |   0.80
citation   |   0.10 |   0.40 |   0.50
format     |   0.20 |   0.70 |  -0.10
tool_use   |   0.00 |   0.30 |   0.40
verbosity  |  -0.10 |  -0.20 |   0.30


## Baseline / 基线：直接平均三个微调模型

模型平均等价于平均三个 delta。它控制了总体范数，但相反符号会相互抵消，例如 friendly 与 invoice 对 brevity 的目标相反。

In [2]:
stacked = np.stack(list(deltas.values()))  # 把三个 task delta 堆叠为任务乘参数矩阵
average_delta = stacked.mean(axis=0)  # 计算直接模型平均得到的合并变化
average_model = base + average_delta  # 把平均 delta 应用到共同 base
sign_conflicts = [parameter_names[index] for index in range(len(parameter_names)) if np.any(stacked[:, index] > 0) and np.any(stacked[:, index] < 0)]  # 找出同时存在正负更新的坐标
print("直接平均 delta：")  # 输出基线合并后的每个可解释参数
for name, value in zip(parameter_names, average_delta):  # 逐坐标展示平均结果
    print(f"{name:10} | {value:6.3f}")  # 观察冲突更新如何被抵消
print("存在符号冲突的参数：", sign_conflicts)  # 明确输出需要 TIES 处理的坐标


直接平均 delta：
tone       |  0.400
brevity    |  0.100
invoice_id |  0.233
amount     |  0.267
risk       |  0.367
refusal    |  0.233
citation   |  0.333
format     |  0.267
tool_use   |  0.233
verbosity  | -0.000
存在符号冲突的参数： ['brevity', 'invoice_id', 'refusal', 'format', 'verbosity']


## 核心实现：Task Arithmetic、TIES 与 DARE

Task Arithmetic 直接相加；TIES 按每个 Adapter 的绝对值中位数裁剪，再用坐标总和决定主导符号；DARE 用固定随机种子丢弃一半更新并除以保留率重标定。

In [3]:
task_arithmetic_delta = stacked.sum(axis=0)  # 使用缩放系数一直接相加三个任务变化
def ties_merge(task_deltas, keep_fraction=0.5):  # 从零实现裁剪、符号选举和同向合并
    trimmed = np.zeros_like(task_deltas)  # 初始化裁剪后的任务矩阵
    keep_count = max(1, int(np.ceil(task_deltas.shape[1] * keep_fraction)))  # 计算每个任务保留的最大变化坐标数
    for row_index, row in enumerate(task_deltas):  # 逐 Adapter 进行绝对值裁剪
        keep_indices = np.argsort(np.abs(row))[-keep_count:]  # 选择绝对变化最大的坐标
        trimmed[row_index, keep_indices] = row[keep_indices]  # 只保留任务最重要的更新
    elected_sign = np.sign(trimmed.sum(axis=0))  # 使用所有保留更新的代数和选举主导符号
    merged = np.zeros(task_deltas.shape[1])  # 初始化 TIES 合并结果
    for column in range(task_deltas.shape[1]):  # 逐坐标合并与主导符号一致的更新
        matching = trimmed[:, column][np.sign(trimmed[:, column]) == elected_sign[column]]  # 过滤反向冲突和零更新
        merged[column] = matching.mean() if len(matching) else 0.0  # 对同向更新取平均避免随任务数放大
    return merged, trimmed, elected_sign  # 返回合并 delta 和可审计中间量
ties_delta, ties_trimmed, elected_sign = ties_merge(stacked)  # 对三个 Adapter 执行 TIES 合并
rng = np.random.default_rng(2506)  # 固定 DARE 稀疏掩码保证输出可复现
drop_probability = 0.5  # 定义丢弃一半 task delta 的教学比例
mask = rng.random(stacked.shape) >= drop_probability  # 为每个任务坐标生成确定性保留掩码
dare_rescaled = stacked * mask / (1 - drop_probability)  # 对保留更新除以保留率维持期望幅度
dare_delta = dare_rescaled.mean(axis=0)  # 平均三个稀疏重标定 Adapter
print("TIES 中间量：parameter | elected_sign | kept_values | merged")  # 输出裁剪与符号选举机制
for index, name in enumerate(parameter_names):  # 逐参数展示保留更新和最终值
    print(f"{name:10} | {int(elected_sign[index]):2} | {ties_trimmed[:, index].tolist()} | {ties_delta[index]:.3f}")  # 观察冲突项如何被过滤
print("DARE 每个 Adapter 保留坐标数：", mask.sum(axis=1).tolist())  # 展示随机稀疏化实际结果


TIES 中间量：parameter | elected_sign | kept_values | merged
tone       |  1 | [0.8, 0.0, 0.0] | 0.800
brevity    |  1 | [0.6, -0.5, 0.0] | 0.600
invoice_id |  1 | [0.0, 0.9, 0.0] | 0.900
amount     |  1 | [0.0, 0.8, 0.0] | 0.800
risk       |  1 | [0.2, 0.0, 0.9] | 0.550
refusal    |  1 | [-0.2, 0.0, 0.8] | 0.800
citation   |  1 | [0.0, 0.4, 0.5] | 0.450
format     |  1 | [0.2, 0.7, 0.0] | 0.450
tool_use   |  1 | [0.0, 0.0, 0.4] | 0.400
verbosity  |  1 | [0.0, 0.0, 0.3] | 0.300
DARE 每个 Adapter 保留坐标数： [7, 6, 3]


## 失败案例与修正：未缩放 Task Arithmetic 造成范数爆炸

三个 delta 直接相加会让共享正向坐标累积，整体范数可能超过任何单一 Adapter。修正可以校准缩放系数，并在发布前设置 delta norm 与安全探针门禁。

In [4]:
adapter_norms = {name: float(np.linalg.norm(delta)) for name, delta in deltas.items()}  # 计算三个单任务 Adapter 的变化范数
unsafe_norm = float(np.linalg.norm(task_arithmetic_delta))  # 计算未缩放 Task Arithmetic 范数
norm_limit = max(adapter_norms.values()) * 1.1  # 使用最大单任务范数的一点一倍作为教学发布上限
safe_scale = min(1.0, norm_limit / max(unsafe_norm, 1e-12))  # 计算不超过范数门禁的缩放系数
safe_task_delta = task_arithmetic_delta * safe_scale  # 对合并变化应用校准缩放
safe_norm = float(np.linalg.norm(safe_task_delta))  # 验证修正后的总体范数
unsafe_publish = unsafe_norm <= norm_limit  # 判断未缩放模型是否能通过门禁
safe_publish = safe_norm <= norm_limit + 1e-12  # 判断缩放模型是否可进入后续评测
print("单 Adapter 范数：", {name: round(value, 3) for name, value in adapter_norms.items()})  # 展示发布上限的依据
print(f"未缩放 Task Arithmetic：norm={unsafe_norm:.3f}，limit={norm_limit:.3f}，publish={unsafe_publish}")  # 展示范数放大失败
print(f"缩放后：scale={safe_scale:.3f}，norm={safe_norm:.3f}，publish={safe_publish}")  # 展示门禁修正结果


单 Adapter 范数： {'friendly_support': 1.068, 'invoice_extract': 1.581, 'fraud_caution': 1.459}
未缩放 Task Arithmetic：norm=2.548，limit=1.739，publish=False
缩放后：scale=0.683，norm=1.739，publish=True


## 结果表：四种合并方案的任务方向与参数风险

In [5]:
def cosine(left, right):  # 计算合并 delta 与单任务方向的一致性
    denominator = np.linalg.norm(left) * np.linalg.norm(right)  # 计算两个向量范数乘积
    return float(left @ right / denominator) if denominator > 0 else 0.0  # 返回稳定余弦相似度
schemes = {"average": average_delta, "task_arithmetic_safe": safe_task_delta, "ties": ties_delta, "dare": dare_delta}  # 汇总四种可比较合并变化
print("scheme | norm | friendly_cos | invoice_cos | fraud_cos | mean_cos")  # 输出任务方向保留与参数范数
scheme_metrics = {}  # 保存各方案指标供发布回归
for name, delta in schemes.items():  # 逐方案计算三个任务探针
    similarities = [cosine(delta, task_delta) for task_delta in deltas.values()]  # 计算与三个 Adapter 的方向相似度
    mean_similarity = sum(similarities) / len(similarities)  # 汇总平均任务方向保留度
    scheme_metrics[name] = {"norm": float(np.linalg.norm(delta)), "mean_cos": mean_similarity}  # 保存范数和平均相似度
    print(f"{name:20} | {np.linalg.norm(delta):.3f} | {similarities[0]:.3f} | {similarities[1]:.3f} | {similarities[2]:.3f} | {mean_similarity:.3f}")  # 展示能力与风险权衡
print("TIES 非零参数：", [name for name, value in zip(parameter_names, ties_delta) if value != 0])  # 展示裁剪后的可审计稀疏结构


scheme | norm | friendly_cos | invoice_cos | fraud_cos | mean_cos
average              | 0.849 | 0.544 | 0.616 | 0.680 | 0.613
task_arithmetic_safe | 1.739 | 0.544 | 0.616 | 0.680 | 0.613
ties                 | 2.012 | 0.491 | 0.586 | 0.594 | 0.557
dare                 | 1.352 | 0.762 | 0.381 | 0.324 | 0.489
TIES 非零参数： ['tone', 'brevity', 'invoice_id', 'amount', 'risk', 'refusal', 'citation', 'format', 'tool_use', 'verbosity']


## 结果解读

直接平均保留了三个任务的共同方向，但在 brevity、invoice_id、refusal 等冲突坐标上发生抵消。TIES 明确展示了哪些更新被裁剪、主导符号如何选择；DARE 的结果依赖随机掩码，必须用多种种子评估。未缩放 Task Arithmetic 未通过范数门禁，说明“把 delta 相加”不是可直接发布的模型。

## 生产边界

真实合并需处理不同 tokenizer、架构、参数命名、量化和训练 base 不一致。参数余弦只是诊断代理，最终必须在各任务、通用能力、安全、幻觉和拒答数据上评测，并与多任务训练比较。DARE 需要多随机种子，TIES 的裁剪比例和缩放系数不能只在测试集调优。

## 最小回归测试

In [6]:
assert len(deltas) == 3 and len(parameter_names) == 10  # 保证案例包含三个业务 Adapter 和可读参数方向
assert len(sign_conflicts) > 0  # 保证输入确实包含需要处理的符号冲突
assert unsafe_publish is False  # 保证未缩放 Task Arithmetic 触发范数风险
assert safe_publish is True  # 保证缩放后的合并变化通过教学范数门禁
assert np.isfinite(ties_delta).all() and np.isfinite(dare_delta).all()  # 保证 TIES 与 DARE 不产生无效参数
assert all(np.isfinite(metric["mean_cos"]) for metric in scheme_metrics.values())  # 保证四种方案的任务方向指标可比较
